# Notebook 1: Loan Overview (Simple - Pure SQL)
Basic SQL queries against `LOAN_DB.LENDING` to explore loan data.

In [ ]:
%%sql -r portfolio_totals
SELECT COUNT(*) AS TOTAL_LOANS,
       SUM(LOAN_AMOUNT) AS TOTAL_PORTFOLIO_VALUE,
       AVG(LOAN_AMOUNT) AS AVG_LOAN_SIZE,
       AVG(INTEREST_RATE) AS AVG_INTEREST_RATE
FROM LOAN_DB.LENDING.LOANS

In [ ]:
import anthropic

client = anthropic.Anthropic()

row = portfolio_totals.iloc[0]
total_loans = int(row['TOTAL_LOANS'])
total_value = float(row['TOTAL_PORTFOLIO_VALUE'])
avg_loan_size = float(row['AVG_LOAN_SIZE'])
avg_interest_rate = float(row['AVG_INTEREST_RATE'])

prompt = f"""
Here are the top-level portfolio totals from a lending database:

- Total Loans: {total_loans}
- Total Portfolio Value: ${total_value:,.2f}
- Average Loan Size: ${avg_loan_size:,.2f}
- Average Interest Rate: {avg_interest_rate:.2f}%

Please:
1. Assess whether the average loan size and interest rate seem typical for a consumer/commercial lending portfolio.
2. Comment on what the total portfolio size implies about the scale of this lender.
3. Flag any headline metric that warrants a closer look or follow-up analysis.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Portfolio Totals")
print("=" * 60)
print(message.content[0].text)

In [ ]:
%%sql -r loans_by_type
SELECT LOAN_TYPE,
       COUNT(*) AS LOAN_COUNT,
       SUM(LOAN_AMOUNT) AS TOTAL_AMOUNT,
       ROUND(AVG(INTEREST_RATE), 2) AS AVG_RATE
FROM LOAN_DB.LENDING.LOANS
GROUP BY LOAN_TYPE
ORDER BY TOTAL_AMOUNT DESC

In [ ]:
import anthropic

client = anthropic.Anthropic()

records = loans_by_type.to_dict(orient='records')
total_portfolio = sum(r['TOTAL_AMOUNT'] for r in records)
for r in records:
    r['PCT_OF_PORTFOLIO'] = round(r['TOTAL_AMOUNT'] / total_portfolio * 100, 1)

prompt = f"""
Below is a breakdown of loans grouped by type, ordered by total amount descending:

{records}

Each record has: LOAN_TYPE, LOAN_COUNT, TOTAL_AMOUNT, AVG_RATE, PCT_OF_PORTFOLIO.

Please:
1. Identify the dominant loan type(s) and assess whether the concentration is a risk.
2. Check whether interest rates are sensibly differentiated across loan types (e.g. unsecured loans should typically carry higher rates than secured ones).
3. Highlight any loan type with an unusual count-to-amount ratio or rate that deserves further investigation.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Loans by Type")
print("=" * 60)
print(message.content[0].text)

In [ ]:
%%sql -r status_summary
SELECT LOAN_STATUS,
       COUNT(*) AS COUNT,
       SUM(LOAN_AMOUNT) AS TOTAL_EXPOSURE
FROM LOAN_DB.LENDING.LOANS
GROUP BY LOAN_STATUS
ORDER BY COUNT DESC

In [ ]:
import anthropic

client = anthropic.Anthropic()

records = status_summary.to_dict(orient='records')
total_exposure = sum(r['TOTAL_EXPOSURE'] for r in records)
total_count = sum(r['COUNT'] for r in records)
for r in records:
    r['PCT_COUNT'] = round(r['COUNT'] / total_count * 100, 1)
    r['PCT_EXPOSURE'] = round(r['TOTAL_EXPOSURE'] / total_exposure * 100, 1)

prompt = f"""
Below is a loan status summary showing count and total dollar exposure per status:

{records}

Each record has: LOAN_STATUS, COUNT, TOTAL_EXPOSURE, PCT_COUNT (% of total loans), PCT_EXPOSURE (% of total dollars).

Please:
1. Evaluate the overall portfolio health — what proportion of loans and exposure are in good vs. at-risk statuses?
2. Flag any statuses (e.g. default, charged-off, late) that represent a disproportionate share of dollar exposure vs. loan count.
3. Provide a brief risk assessment and suggest what management action, if any, this distribution warrants.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Loan Status Health")
print("=" * 60)
print(message.content[0].text)

In [ ]:
%%sql -r top_states
SELECT c.STATE,
       COUNT(l.LOAN_ID) AS LOAN_COUNT,
       SUM(l.LOAN_AMOUNT) AS TOTAL_LENT
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
GROUP BY c.STATE
ORDER BY TOTAL_LENT DESC
LIMIT 10

In [ ]:
import anthropic

client = anthropic.Anthropic()

records = top_states.to_dict(orient='records')
total_lent = sum(r['TOTAL_LENT'] for r in records)
for r in records:
    r['PCT_OF_TOP10'] = round(r['TOTAL_LENT'] / total_lent * 100, 1)
    r['AVG_LOAN_SIZE'] = round(r['TOTAL_LENT'] / r['LOAN_COUNT'], 2)

prompt = f"""
Below are the top 10 US states by total lending volume (this may not represent the full portfolio):

{records}

Each record has: STATE, LOAN_COUNT, TOTAL_LENT, PCT_OF_TOP10 (share within this top-10 list), AVG_LOAN_SIZE.

Please:
1. Comment on the geographic concentration — are 1-2 states dominating, and is that a risk?
2. Compare average loan sizes across states — do high-volume states also have larger average loans, or is volume driven by count?
3. Note any surprising states in the top 10 (unexpectedly high or absent) and what that might suggest about the lender's market strategy.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Top States by Lending Volume")
print("=" * 60)
print(message.content[0].text)